# Loan Approvals

In [9]:
!pip cache purge --quiet

In [10]:
import base64
import pandas as pd
import mimetypes
import numpy as np

from IPython.display import HTML

In [11]:
# Configuration
n_rows = 1000
np.random.seed(42)

# Generate base features
annual_income = np.random.randint(20000, 200000, n_rows)
employment_length = np.random.choice(range(0, 21), n_rows)

# Debt-to-Income (DTI) with some noise
base_dti = np.random.uniform(0, 40, n_rows)
dti = np.clip(base_dti - (annual_income / 200000) * 10 + np.random.normal(0, 3, n_rows), 0, 100)

# Interest rate inversely related to income
interest_rate = np.round(np.random.uniform(5, 30, n_rows) - (annual_income / 200000) * 5, 2)
interest_rate = np.clip(interest_rate, 5, 30)

# Open credit lines and revolving utilization
open_credit_lines = np.random.randint(1, 20, n_rows)
revol_util = np.round(np.random.uniform(0, 100, n_rows), 2)

# Home ownership and loan purpose
home_ownership = np.random.choice(
    ["rent", "own", "mortgage", "other"], n_rows, p = [0.35, 0.25, 0.35, 0.05]
)
loan_purpose = np.random.choice(
    ["debt_consolidation", "credit_card", "home_improvement",
     "major_purchase", "small_business", "car", "medical",
     "vacation", "wedding", "other"], n_rows
)

# Loan term
term = np.random.choice([36, 60], n_rows)

# LoanAmount proportional to income + noise + mild outliers
loan_amount = np.round(annual_income * np.random.uniform(0.05, 0.25, n_rows))
loan_amount = np.clip(loan_amount + np.random.normal(0, 5000, n_rows), 1000, 50000)

# Introduce extreme outliers (5% of data)
n_outliers = int(0.05 * n_rows)
outlier_indices = np.random.choice(np.arange(n_rows), n_outliers, replace = False)
for idx in outlier_indices:
    if np.random.rand() < 0.5:
        loan_amount[idx] = min(int(loan_amount[idx] * np.random.uniform(1.5, 3)), 50000)
    else:
        loan_amount[idx] = max(int(loan_amount[idx] * np.random.uniform(0.5, 0.8)), 1000)

# Issue date
issue_date = pd.to_datetime(
    np.random.randint(
        pd.Timestamp("2015-01-01").value // 10**9,
        pd.Timestamp("2023-12-31").value // 10**9,
        n_rows
    ),
    unit = "s"
)

# Monthly income
monthly_income = np.round(annual_income / 12, 2)

# Temporal risk factor
date_norm = (issue_date - issue_date.min()) / (issue_date.max() - issue_date.min())
temporal_risk = 0.1 * date_norm

# Interaction features
high_income_low_dti = ((annual_income > 80000) & (dti < 20)).astype(int)
interest_impact = interest_rate * loan_amount / (annual_income + 1)

# Risk score
risk_score = (
    0.25 * (dti / 40) +
    0.2  * (interest_rate / 30) +
    0.15 * (loan_amount / 50000) +
    0.1  * (1 - employment_length / 20) +
    0.1  * temporal_risk +
    0.1  * high_income_low_dti +
    0.1  * (interest_impact / (interest_impact.max() + 1e-6))
)

prob_approved = 1 - risk_score
prob_approved = np.clip(prob_approved, 0, 1)

# Loan status with stochasticity
loan_status = np.random.binomial(1, prob_approved)

# Construct DataFrame
df = pd.DataFrame({
    "ID": np.arange(1, n_rows + 1),
    "IssueDate": issue_date,
    "AnnualIncome": annual_income,
    "MonthlyIncome": monthly_income,
    "LoanAmount": loan_amount,
    "Term": term,
    "InterestRate": interest_rate,
    "EmploymentLength": employment_length,
    "HomeOwnership": home_ownership,
    "LoanPurpose": loan_purpose,
    "DebtToIncome": np.round(dti, 2),
    "OpenCreditLines": open_credit_lines,
    "RevolUtilization": np.round(revol_util, 2),
    "HighIncome_LowDTI": high_income_low_dti,
    "InterestImpact": interest_impact,
    "LoanStatus": loan_status
})

# Sort by IssueDate
df = df.sort_values("IssueDate").reset_index(drop = True)

In [12]:
df.head()

,ID,IssueDate,AnnualIncome,MonthlyIncome,LoanAmount,Term,InterestRate,EmploymentLength,HomeOwnership,LoanPurpose,DebtToIncome,OpenCreditLines,RevolUtilization,HighIncome_LowDTI,InterestImpact,LoanStatus
0,849,2015-01-01 18:06:37,43624,3635.33,8257.920192,60,19.42,13,rent,other,29.13,10,79.73,0,3.676076,0
1,378,2015-01-03 16:53:17,97575,8131.25,14820.090985,60,12.51,20,mortgage,major_purchase,3.76,8,99.80,1,1.900051,0
2,889,2015-01-06 21:33:10,66271,5522.58,12601.890791,60,16.99,11,rent,car,3.32,17,68.52,0,3.230718,1
3,141,2015-01-17 08:36:27,95450,7954.17,1387.446055,60,8.04,8,own,vacation,16.53,1,22.31,1,0.116867,1
4,915,2015-01-18 13:17:40,43275,3606.25,1688.485297,60,28.58,4,rent,car,0.00,3,11.24,0,1.115096,0


In [13]:
loans_csv = "loans.csv"

df.to_csv(loans_csv, index = False)

In [14]:
mime_type, _ = mimetypes.guess_type(loans_csv)

with open(loans_csv, "rb") as f:
    data = f.read()
b64 = base64.b64encode(data).decode()
href = f'<a download="{loans_csv}" href="data:{mime_type};base64,{b64}">Download {loans_csv}</a>'
HTML(href)